In [1]:
using Lux, LuxCUDA, MLUtils
using Optimisers, Random, Statistics
using Zygote
using DiffEqFlux, OrdinaryDiffEq
using Images, JLD2
using ComponentArrays
using DeepEquilibriumNetworks
using Plots
using Dates

using SciMLSensitivity, Lux, NonlinearSolve, LinearSolve

In [2]:
const cdev = cpu_device()
const gdev = gpu_device()
dev = gdev

(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

In [3]:
model_first_conv = Chain(
        
        Conv((5, 5), 1 => 8, tanh; pad = 2),
        Conv((5, 5), 8 => 16; pad = 2),
        SkipConnection(Chain(
            Conv((1, 1), 16 => 64, gelu),
            Conv((1, 1), 64 => 32, gelu),
            Conv((1, 1), 32 => 16),
        ), +),
    ) 
model_conv = Chain(
        Parallel(+, 
            NoOpLayer(), # Pass z through
            NoOpLayer()  # Pass x through
        ),
        model_first_conv,
        GroupNorm(16, 4),
        Conv((5, 5), 16 => 32, tanh; pad = 2),
        Conv((5, 5), 32 => 64; pad = 2),
        SkipConnection(Chain(
            Conv((1, 1), 64 => 128, gelu),
            Conv((1, 1), 128 => 128, gelu),
            Conv((1, 1), 128 => 64, gelu),
        ), +),
        GroupNorm(64, 4),
        Conv((1, 1), 64 => 64, gelu),
        Conv((1, 1), 64 => 32, gelu),
        Conv((1, 1), 32 => 1),
    ) 


Chain(
    layer_1 = Parallel(
        connection = +,
        layer_(1-2) = NoOpLayer(),
    ),
    layer_2 = Chain(
        layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
        layer_2 = Conv((5, 5), 8 => 16, pad=2),   # 3_216 parameters
        layer_3 = SkipConnection(
            connection = +,
            layers = Chain(
                layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
            ),
        ),
    ),
    layer_3 = GroupNorm(16, 4, affine=true),      # 32 parameters
    layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
    layer_5 = Conv((5, 5), 32 => 64, pad=2),      # 51_264 parameters
    layer_6 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 64 => 128, gelu_tanh),  # 8_320 parameters
            layer_2 = Con

In [5]:
deq = DeepEquilibriumNetwork(model_conv, NewtonRaphson(; linsolve=KrylovJL_GMRES()); init = nothing, verbose=false,
        linsolve_kwargs=(; maxiters=10), maxiters=10)
model = SkipConnection(connection = +, layers = deq)

SkipConnection(
    connection = +,
    layers = DeepEquilibriumNetwork(
        model = Chain(
            layer_1 = Parallel(
                connection = +,
                layer_(1-2) = NoOpLayer(),
            ),
            layer_2 = Chain(
                layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                layer_3 = SkipConnection(
                    connection = +,
                    layers = Chain(
                        layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                        layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                        layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                    ),
                ),
            ),
            layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
            layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
            layer_5 = C

In [36]:
model = DeepEquilibriumNetwork(model_conv, Tsit5(); save_everystep = false, sensealg = BacksolveAdjoint(; autojacvec = ZygoteVJP()), reltol = 1e-3, abstol = 1e-4, save_start = false)

DeepEquilibriumNetwork(
    model = Chain(
        layer_1 = Parallel(
            connection = +,
            layer_(1-2) = NoOpLayer(),
        ),
        layer_2 = Chain(
            layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
            layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
            layer_3 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                    layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                    layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                ),
            ),
        ),
        layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
        layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
        layer_5 = Conv((5, 5), 32 => 64, pad=2),  # 51_264 parameters
        layer_6 = SkipConnection(
            connection = +,
            laye

In [12]:
#@load "ps_latest.jld2" ps
#ps_node = ps
rng = Xoshiro(0)
ps , st = Lux.setup(rng, model)
#ps.model = ps_node

((model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = (weight = Float32[-0.5096706 0.31564143 … 0.5504109 0.21083479; -0.051686242 -0.44722694 … 0.42996323 -0.46218252; … ; 0.055012714 -0.10226735 … 0.13269973 -0.31160903; 0.21374741 -0.5431161 … 0.19860923 0.31442145;;;; 0.15156177 5.093088f-6 … -0.3233329 0.0361124; -0.18664144 -0.03434386 … 0.20643677 0.2529072; … ; 0.3659556 -0.44471633 … 0.38821638 0.10493441; 0.22864234 -0.044965085 … 0.07534941 -0.098589316;;;; 0.42701256 0.0175333 … -0.10607519 -0.2367328; -0.13226888 -0.35014457 … -0.060714014 -0.5477423; … ; -0.30516297 0.3741171 … -0.44158098 0.50796825; 0.5141952 0.1775339 … -0.42729515 0.14101206;;;; 0.11648312 -0.20503472 … 0.2995325 -0.14051251; -0.22026113 -0.10612598 … -0.30030417 -0.54208064; … ; -0.36221325 0.2780396 … 0.30802768 0.381799; -0.26694658 0.29389465 … 0.5221397 -0.45112377;;;; -0.20194073 0.47159848 … 0.25676164 -0.0026746283; 0.37886196 0.14893022 … 0.35421994 0.027

In [7]:
ps = ps |> ComponentArray |> dev
st = st |> dev

(model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer_3 = NamedTuple(), layer_4 = NamedTuple(), layer_5 = NamedTuple(), layer_6 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple()), layer_7 = NamedTuple(), layer_8 = NamedTuple(), layer_9 = NamedTuple(), layer_10 = NamedTuple()), fixed_depth = Val{0}(), init = NamedTuple(), solution = DeepEquilibriumSolution
 * Initial Guess: nothing
 * Steady State: nothing
 * Residual: nothing
 * Jacobian Loss: nothing
 * NFE: 0, rng = Xoshiro(0x4b2aef8cf38557c9, 0x5da23fce547f84b5, 0x22808ee13fe83c01, 0xfd378ad274a3ec26, 0x22a21880af5dc689))

In [8]:
opt = Optimisers.NAdam(1e-4)
state = Optimisers.setup(opt,ps)
train_state = Lux.Training.TrainState(model, ps, st, opt)

TrainState(
    SkipConnection(
        connection = +,
        layers = DeepEquilibriumNetwork(
            model = Chain(
                layer_1 = Parallel(
                    connection = +,
                    layer_(1-2) = NoOpLayer(),
                ),
                layer_2 = Chain(
                    layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                    layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                    layer_3 = SkipConnection(
                        connection = +,
                        layers = Chain(
                            layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                            layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                            layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                        ),
                    ),
                ),
                layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
       

In [9]:
x = randn(Float32,128,128,1,4)
y_true = randn(Float32,128,128,1,4)
x_dev = x |> dev
y_dev = y_true |> dev
y, _ = model(x_dev, ps, st)

FieldError: FieldError: type NamedTuple has no field `model`, available fields: `layer_1`, `layer_2`, `layer_3`, `layer_4`, `layer_5`, `layer_6`, `layer_7`, `layer_8`, `layer_9`

In [41]:
y

128×128×1×4 CuArray{Float32, 4, CUDA.DeviceMemory}:
[:, :, 1, 1] =
 -0.0895817  -0.0568841   0.181138   …  -0.147395    0.343783    0.291161
  0.154047   -0.0825094   0.238452       0.599491   -0.0275553   0.0402318
  0.0858208   0.478512    0.693986       0.214729    0.274355    0.241446
  0.292227    0.367953    0.841573       0.172951    0.18309     0.517982
  0.644021    0.16049     0.344857       0.357439    0.862105    0.384632
  0.172303    0.0345062   0.58676    …  -0.0942485   0.11895     0.349348
  0.258731    0.0165451   0.020336       0.324327   -0.250083    0.270267
  0.666988    0.304665    0.472683       0.319562    0.905905    0.590722
  1.04396     0.257696    0.152014      -0.32125     0.366792    0.158876
  0.713751    0.575919    0.104247       0.141188    0.579793    0.125683
  ⋮                                  ⋱   ⋮                      
  0.682844   -0.116071    0.601245       0.310276    0.616173   -0.00434607
  0.135609    0.757396   -0.0306996  …   0.54727   